# SemMap · ConceptNet 1M: параллельный пилот, CUDA в приоритете

Сравниваем 100k и 1M при равных бюджетах 500/1000/3000/5000 ego-центров. CPU-процессы параллельно читают официальный дамп и считают типизированные WL-отпечатки; CPU-потоки извлекают ego. Точный VF2 остаётся CPU. Для отдельного Wishart-этапа k=4/5 CUDA выбирается автоматически, а фактический backend обязательно проверяется. При отсутствии GPU режим auto явно переключается на CPU; режим cuda требует GPU.

Это **пилот словаря прототипов**, не полное сжатие миллиона вершин. Предварительный порог RAM 36 GiB, локального диска 16 GiB; пороги не гарантируют отсутствие OOM. Отчёты сохраняются отдельно на Drive. Исходные runs не меняются.

In [ ]:
# 1. Настройки и проверка ресурсов.
import os,sys,json,shutil,subprocess,hashlib,tempfile
from pathlib import Path
from datetime import datetime,timezone
DRIVE_ROOT=Path('/content/drive/MyDrive/SemanticMap/colab/wishart')
DATA=DRIVE_ROOT/'data'
TSV100=DATA/'conceptnet_en_100k.tsv'
TSV1M=DATA/'conceptnet_en_1m.tsv'
DUMP=DATA/'conceptnet-assertions-5.7.0.csv.gz'
OFFICIAL_URL='https://s3.amazonaws.com/conceptnet/downloads/2019/edges/conceptnet-assertions-5.7.0.csv.gz'
DOWNLOAD_IF_MISSING=True
RUN_100K=True
BUDGETS=(500,1000,3000,5000)
SAMPLE_CENTERS=5000
MAX_EGO_NODES=48
SEED=1729
CPU_WORKERS=min(4,max(1,os.cpu_count() or 1))
DEVICE='auto'  # auto: prefer CUDA; cuda: require it; cpu: force CPU
GPU_BATCH_SIZE=128
MIN_RAM_GIB=36
MIN_DISK_GIB=16
REPO_URL='https://github.com/SemanticMap/semgraphex.git'
BRANCH='feature/conceptnet-1m-cuda-port-pilot'
REPO=Path('/content/semgraphex-million-pilot')
SCRATCH=Path('/content/semmap-million-pilot')
INPUT=SCRATCH/'input'
OUTPUT=SCRATCH/'analysis'
DEST=DRIVE_ROOT/'analysis'/'conceptnet-1m-port-cuda-pilot-v2'
for p in (SCRATCH,INPUT,OUTPUT):p.mkdir(parents=True,exist_ok=True)
for name in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ[name]='1'
mem_available=next(int(line.split()[1])*1024 for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemAvailable:'))
cg=Path('/sys/fs/cgroup/memory.max')
if cg.is_file() and cg.read_text().strip().isdigit():
    mem_available=min(mem_available,int(cg.read_text().strip())-int(Path('/sys/fs/cgroup/memory.current').read_text().strip()))
print({'python':sys.version.split()[0],'cpu_workers':CPU_WORKERS,'available_ram_gib':round(mem_available/1024**3,1),'scratch_free_gib':round(shutil.disk_usage(SCRATCH).free/1024**3,1),'device':DEVICE})
if mem_available<MIN_RAM_GIB*1024**3:raise MemoryError('Нужен High-RAM runtime или более экономный extractor.')
if shutil.disk_usage(SCRATCH).free<MIN_DISK_GIB*1024**3:raise OSError('Недостаточно места на /content.')

In [ ]:
# 2. Монтирование Drive и клонирование новой ветки.
from google.colab import drive
drive.mount('/content/drive')
if RUN_100K and not TSV100.is_file():raise FileNotFoundError(TSV100)
if REPO.exists():shutil.rmtree(REPO)
probe=subprocess.run(['git','ls-remote','--heads',REPO_URL,BRANCH],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
env=os.environ.copy()
askpass=None
if probe.returncode!=0:
    from google.colab import userdata
    token=userdata.get('GITHUB_TOKEN')
    if not token:raise RuntimeError('Добавьте read-only GITHUB_TOKEN в Colab Secrets и разрешите доступ этому блокноту.')
    fd,name=tempfile.mkstemp(prefix='semmap-git-',suffix='.sh');os.close(fd)
    askpass=Path(name)
    askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) echo x-access-token;;\n*Password*) echo "$GITHUB_TOKEN";;\nesac\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass),GIT_TERMINAL_PROMPT='0',GITHUB_TOKEN=token)
try:
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(REPO)],check=True,env=env)
finally:
    if askpass:askpass.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN',None)
CODE_SHA=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
subprocess.run([sys.executable,'-m','pip','install','-q','-c',str(REPO/'requirements/constraints-colab.txt'),'-e',str(REPO)+'[wishart,notebook]'],check=True)
pkg_src=str((REPO/'src').resolve())
if pkg_src not in sys.path:sys.path.insert(0,pkg_src)
import importlib
importlib.invalidate_caches()
from semmap_haken.million_port_pilot import sha256_file
from semmap_haken.wishart_gpu import resolve_device,cuda_diagnostics
selected_device=resolve_device(DEVICE)
gpu_name=None
if selected_device=='cuda':
    import torch
    gpu_name=torch.cuda.get_device_name(0)
print({'git':CODE_SHA,'device_selected':selected_device,'gpu':gpu_name,'cuda_available':cuda_diagnostics()['cuda_available']})

In [ ]:
# 3. Готовый 1M TSV или трёхпроходное выделение из официального ConceptNet.
LOCAL_1M=INPUT/'conceptnet_en_1m.tsv'
if TSV1M.is_file():
    shutil.copy2(TSV1M,LOCAL_1M)
else:
    local_dump=INPUT/'conceptnet-assertions-5.7.0.csv.gz'
    if DUMP.is_file():
        shutil.copy2(DUMP,local_dump)
    elif DOWNLOAD_IF_MISSING:
        import urllib.request
        with urllib.request.urlopen(OFFICIAL_URL,timeout=60) as stream, local_dump.open('wb') as out:
            shutil.copyfileobj(stream,out,length=4*1024*1024)
    else:raise FileNotFoundError('Нет готового 1M TSV или официального дампа на Drive.')
    if local_dump.stat().st_size<1000000:raise IOError('Слишком маленький файл ConceptNet.')
    extract_metadata=INPUT/'extract_1m_metadata.json'
    subprocess.run([
        sys.executable,str(REPO/'scripts/extract_conceptnet_100k.py'),
        '--input',str(local_dump),'--output',str(LOCAL_1M),'--metadata',str(extract_metadata),
        '--target-nodes','1000000','--candidate-multiplier','3',
        '--workers',str(CPU_WORKERS),'--batch-size','10000','--min-weight','1.0',
        '--relations','RelatedTo','IsA','PartOf','HasA','UsedFor','HasProperty','CapableOf','Causes',
    ],check=True)
    check=json.loads(extract_metadata.read_text())
    if check['selected_nodes_seen_in_induced_edges']!=1000000 or check['missing_selected_node_count']!=0:
        raise RuntimeError('Экстрактор не подтвердил 1 000 000 связанных вершин.')
    TSV1M.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(LOCAL_1M,TSV1M)
    if sha256_file(LOCAL_1M)!=sha256_file(TSV1M):raise IOError('Повреждён TSV при копировании в Drive.')
print('1M готов:',LOCAL_1M.stat().st_size,'байт')
LOCAL_100=INPUT/'conceptnet_en_100k.tsv'
if RUN_100K:shutil.copy2(TSV100,LOCAL_100)

In [ ]:
# 4. Изолированные процессы запуска: CPU-пулы завершаются до GPU kNN, RAM между графами освобождается.
import pandas as pd
from IPython.display import display
DATASETS=[('100k',LOCAL_100,100000)] if RUN_100K else []
DATASETS.append(('1m',LOCAL_1M,1000000))
reports={}
for label,dataset,node_count in DATASETS:
    digest=sha256_file(dataset)
    dest=DEST/label
    out=OUTPUT/label
    identity={'dataset_sha256':digest,'target_nodes':node_count,'sample_centers':SAMPLE_CENTERS,
              'budgets':list(BUDGETS),'seed':SEED,'max_ego_nodes':MAX_EGO_NODES,
              'workers':CPU_WORKERS,'device_requested':DEVICE,'device_selected':selected_device,
              'code_sha':CODE_SHA}
    if (dest/'COMPLETED').is_file():
        saved=json.loads((dest/'notebook_manifest.json').read_text())
        if not all(saved.get(k)==v for k,v in identity.items()):
            raise RuntimeError('В целевой папке есть другой запуск; измените имя DEST.')
        result=json.loads((dest/'report.json').read_text())
        if selected_device=='cuda' and result['wishart'].get('knn_backend')!='torch_cuda':
            raise RuntimeError('Сохранённый запуск не подтверждает CUDA.')
        reports[label]=result
        print(label,': полный кэш найден.')
        continue
    subprocess.run([
        sys.executable,'-m','semmap_haken.million_port_pilot',
        '--dataset',str(dataset),'--output',str(out),
        '--target-nodes',str(node_count),'--sample-centers',str(SAMPLE_CENTERS),
        '--budgets',*[str(v) for v in BUDGETS],
        '--seed',str(SEED),'--max-ego-nodes',str(MAX_EGO_NODES),
        '--workers',str(CPU_WORKERS),'--device',DEVICE,
        '--gpu-batch-size',str(GPU_BATCH_SIZE),'--source-sha256',digest,
    ],check=True,cwd=REPO,env={**os.environ,'PYTHONPATH':str(REPO/'src')})
    result=json.loads((out/'report.json').read_text())
    assert result['actual_nodes']==node_count and result['dataset_sha256']==digest
    assert all(row['roundtrip_exact'] for row in result['results'])
    if selected_device=='cuda' and result['wishart'].get('knn_backend')!='torch_cuda':
        raise RuntimeError('CUDA была доступна, но kNN не использовал GPU.')
    (out/'notebook_manifest.json').write_text(json.dumps(identity|{'created_utc':datetime.now(timezone.utc).isoformat()},ensure_ascii=False,indent=2)+'\n')
    dest.mkdir(parents=True,exist_ok=True)
    (dest/'COMPLETED').unlink(missing_ok=True)
    for file in out.iterdir():
        if file.is_file() and file.name!='COMPLETED':
            shutil.copy2(file,dest/file.name)
            if file.stat().st_size!=(dest/file.name).stat().st_size:raise IOError('Ошибка Drive sync '+file.name)
    (dest/'COMPLETED').write_text('complete\n')
    reports[label]=result
    print('Сохранено:',dest)
rows=[{'graph':label,**row} for label,report in reports.items() for row in report['results']]
display(pd.DataFrame(rows)[['graph','sample_budget','exact_types','internal_shapes','json_saved_percent','baseline_gzip_bytes','factorized_gzip_bytes','roundtrip_exact','peak_rss_gib']])
for label,report in reports.items():print(label,'parallel:',report['parallel'],'wishart:',report['wishart'])
print('Drive:',DEST)

## Проверка результата

При CPU_WORKERS > 1 ожидается `parallel.fingerprint_backend = process_pool_spawn`; реальное число процессоров указывается в `worker_processes_observed`. При наличии CUDA и DEVICE=auto ожидается `wishart.knn_backend = torch_cuda`. Если CUDA нет, отчёт явно укажет CPU. GPU ускоряет **только kNN typed-WL**, не VF2, не потоковый парсер и не Python-подготовку графа. JSON/gzip относятся к прототипам, а не ко всему графу.